In [2]:
import numpy as np
from thin_film import thin_film_velocities_axisymmetric
import matplotlib.pyplot as plt
from scipy.special import j0, j1, jn_zeros

In [6]:
%matplotlib Qt

In [47]:


N = 128
R = 1.0
h0 = 1.5e-5

dr = R / N
r = (np.arange(N) + 0.5) * dr

eta = np.zeros_like(r)

mode = 10
beta = jn_zeros(0, mode + 1)[mode]
kappa = beta / R

phi = j0(kappa * r)

q, vr, vz, div_flux, F = thin_film_velocities_axisymmetric(
    r,
    eta,
    phi,
    h0,
    R=R,
    side_bc="dirichlet",
    phi_wall=0.0,
)

vr_expected = -kappa * j1(kappa * r)
vz_thin_expected = h0 * kappa**2 * j0(kappa * r)

# Exact finite-depth result for comparison.
vz_exact = kappa * np.tanh(kappa * h0) * j0(kappa * r)

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)
fig.suptitle(f"Thin-film radial velocities for mode {mode} (h0={h0:.1e})")
ax1.plot(r, vr, label="vr thin-film")
ax1.plot(r, vr_expected, "--", label="vr analytic")

ax2.plot(r, vz, label="vz thin-film")
ax2.plot(r, vz_thin_expected, "--", label="vz thin analytic")
ax2.plot(r, vz_exact, ":", label="vz exact finite-depth")

ax1.set_xlabel("r")
ax1.set_ylabel("vr")
ax2.set_xlabel("r")
ax2.set_ylabel("vz")
ax1.legend()
ax2.legend()
plt.show()

print("relative q error vs thin analytic:",
      np.linalg.norm(q - vz_thin_expected) / np.linalg.norm(vz_thin_expected))

print("relative q error vs exact finite-depth:",
      np.linalg.norm(q - vz_exact) / np.linalg.norm(vz_exact))

relative q error vs thin analytic: 0.0062439857978762465
relative q error vs exact finite-depth: 0.006243901340557273


In [18]:
import numpy as np
from dataclasses import dataclass
from typing import Callable, Optional


Array = np.ndarray


@dataclass
class ThinFilmParams:
    R: float = 1.0
    h0: float = 1.5e-5

    # Side-wall condition for the thin-film radial operator.
    # "dirichlet": phi(R) = phi_wall
    # "neumann": phi_r(R) = 0
    side_bc: str = "dirichlet"
    phi_wall: float = 0.0

    # van der Waals term.
    # In the same nondimensional spirit as Roberts-style helium model:
    # V_vdw = vdw_coeff * h0/3 * ((h0/H)^3 - 1)
    use_vdw: bool = True
    vdw_coeff: float = 1.0

    # Optical forcing / energy terms.
    # V_opt = profile(r) * (instant_coeff * I + delayed_coeff * D)
    instant_coeff: float = 0.0
    delayed_coeff: float = 0.0

    # Delayed intensity dynamics:
    # D_t = (I_local - D)/tau
    use_delay: bool = False
    tau_delay: float = 1.0

    # Intensity drive:
    intensity0: float = 0.0
    intensity_amp: float = 0.0
    omega_drive: float = 0.0
    phase_drive: float = 0.0

    # Optical spatial profile.
    # If profile_sigma <= 0, profile = 1 everywhere.
    profile_center: float = 0.0
    profile_sigma: float = 0.0

    # Optional damping directly on phi.
    phi_damping: float = 0.0


def pack_state(r: Array, z: Array, phi: Array, D: Optional[Array] = None) -> Array:
    if D is None:
        return np.concatenate([r, z, phi])
    return np.concatenate([r, z, phi, D])


def unpack_state(y: Array, N: int, use_delay: bool):
    r = y[0:N]
    z = y[N:2*N]
    phi = y[2*N:3*N]

    if use_delay:
        D = y[3*N:4*N]
    else:
        D = None

    return r, z, phi, D


def optical_intensity_scalar(t: float, p: ThinFilmParams) -> float:
    return p.intensity0 + p.intensity_amp * np.sin(p.omega_drive * t + p.phase_drive)


def optical_profile(r: Array, p: ThinFilmParams) -> Array:
    if p.profile_sigma <= 0.0:
        return np.ones_like(r)

    x = (r - p.profile_center) / p.profile_sigma
    return np.exp(-0.5 * x * x)


def finite_volume_faces_from_centers(r: Array, R: float) -> Array:
    """
    Build radial cell faces from possibly nonuniform cell centers.

    Assumes:
        r[0] > 0,
        r is increasing,
        left boundary is 0,
        right boundary is R.
    """
    N = len(r)
    rf = np.empty(N + 1, dtype=float)

    rf[0] = 0.0
    rf[1:N] = 0.5 * (r[:-1] + r[1:])
    rf[N] = R

    return rf


def center_gradient_from_face_gradient(face_grad: Array) -> Array:
    """
    Simple average of neighboring face gradients to cell centers.
    """
    return 0.5 * (face_grad[:-1] + face_grad[1:])


def thin_film_velocities_lagrangian(
    r: Array,
    z: Array,
    phi: Array,
    p: ThinFilmParams,
):
    """
    Compute thin-film velocities for a possibly moving radial grid.

    Returns
    -------
    vr:
        Leading-order radial velocity v_r ~ phi_r.
    vz:
        Vertical velocity at the free surface.
    q:
        Unit-normal velocity.
    z_prime:
        dz/dr at centers.
    div_flux:
        (1/r) d/dr (r H phi_r)
    F:
        Radial film flux at faces, F = r H phi_r.
    """

    r = np.asarray(r, dtype=float)
    z = np.asarray(z, dtype=float)
    phi = np.asarray(phi, dtype=float)

    N = len(r)

    if not (z.shape == r.shape and phi.shape == r.shape):
        raise ValueError("r, z, and phi must have the same shape.")

    if np.any(np.diff(r) <= 0):
        raise ValueError("r must be strictly increasing.")

    if r[0] <= 0.0:
        raise ValueError("This implementation expects cell centers with r[0] > 0.")

    if r[-1] >= p.R:
        raise ValueError("Last cell center must be inside R.")

    H = p.h0 + z

    if np.any(H <= 0.0):
        raise ValueError("Film thickness H = h0 + z became non-positive.")

    rf = finite_volume_faces_from_centers(r, p.R)

    # Face gradient of phi.
    dphi_dr_face = np.zeros(N + 1, dtype=float)

    # Axis regularity.
    # phi_r(0) = 0.
    dphi_dr_face[0] = 0.0

    # Interior faces.
    dphi_dr_face[1:N] = (phi[1:] - phi[:-1]) / (r[1:] - r[:-1])

    # Outer boundary.
    side_bc = p.side_bc.lower()
    if side_bc == "dirichlet":
        dphi_dr_face[N] = (p.phi_wall - phi[-1]) / (p.R - r[-1])
    elif side_bc == "neumann":
        dphi_dr_face[N] = 0.0
    else:
        raise ValueError("side_bc must be 'dirichlet' or 'neumann'.")

    # Thickness at faces.
    H_face = np.empty(N + 1, dtype=float)
    H_face[0] = H[0]
    H_face[1:N] = 0.5 * (H[:-1] + H[1:])
    H_face[N] = H[-1]

    # F = r H phi_r.
    F = rf * H_face * dphi_dr_face

    # Correct finite-volume radial measure:
    # integral r dr over cell = 0.5 * (r_right^2 - r_left^2)
    radial_cell_measure = 0.5 * (rf[1:]**2 - rf[:-1]**2)

    div_flux = (F[1:] - F[:-1]) / radial_cell_measure

    # z_r at centers.
    dz_dr_face = np.zeros(N + 1, dtype=float)
    dz_dr_face[0] = 0.0
    dz_dr_face[1:N] = (z[1:] - z[:-1]) / (r[1:] - r[:-1])

    # For the wall slope, use one-sided extrapolation or simply last interior slope.
    dz_dr_face[N] = dz_dr_face[N - 1]

    z_prime = center_gradient_from_face_gradient(dz_dr_face)

    norm = np.sqrt(1.0 + z_prime * z_prime)

    # Unit-normal velocity.
    q = -div_flux / norm

    # Leading-order radial velocity at centers.
    vr = center_gradient_from_face_gradient(dphi_dr_face)

    # q = (vz - z_r vr) / sqrt(1 + z_r^2)
    # => vz = q sqrt(...) + z_r vr
    vz = q * norm + z_prime * vr

    return vr, vz, q, z_prime, div_flux, F


def vdw_energy_term(z: Array, p: ThinFilmParams) -> Array:
    """
    Nondimensional van der Waals contribution.

    For H = h0 + z:

        V_vdw = h0/3 * [ (h0/H)^3 - 1 ]

    multiplied by p.vdw_coeff.

    This matches the structure h/3 * [1/(1 + Y/h)^3 - 1].
    """

    H = p.h0 + z

    if np.any(H <= 0.0):
        raise ValueError("Film thickness H = h0 + z became non-positive.")

    return p.vdw_coeff * (p.h0 / 3.0) * ((p.h0 / H)**3 - 1.0)


def optical_energy_terms(
    t: float,
    r: Array,
    z: Array,
    phi: Array,
    D: Optional[Array],
    p: ThinFilmParams,
):
    """
    Return optical contribution to phi_t and D_t.

    I_local = profile(r) * I(t)

    V_opt = instant_coeff * I_local + delayed_coeff * D

    D_t = (I_local - D) / tau_delay
    """

    I = optical_intensity_scalar(t, p)
    profile = optical_profile(r, p)
    I_local = profile * I

    if p.use_delay:
        if D is None:
            raise ValueError("p.use_delay=True but D was not provided in the state.")

        D_t = (I_local - D) / p.tau_delay
        V_opt = p.instant_coeff * I_local + p.delayed_coeff * D
    else:
        D_t = None
        V_opt = p.instant_coeff * I_local

    return V_opt, D_t, I_local


def rhs_thin_film_lagrangian(
    t: float,
    y: Array,
    p: ThinFilmParams,
):
    """
    RHS for the state y = [r, z, phi] or y = [r, z, phi, D].

    Equations:
        r_t   = v_r
        z_t   = v_z
        phi_t = 0.5*(v_r^2 + v_z^2)
                + V_vdw(z)
                + V_opt(I,D)
                - phi_damping * phi

        D_t = (I_local - D)/tau_delay, if enabled.
    """

    if p.use_delay:
        if y.size % 4 != 0:
            raise ValueError("For use_delay=True, state must be [r,z,phi,D].")
        N = y.size // 4
    else:
        if y.size % 3 != 0:
            raise ValueError("For use_delay=False, state must be [r,z,phi].")
        N = y.size // 3

    r, z, phi, D = unpack_state(y, N, p.use_delay)

    vr, vz, q, z_prime, div_flux, F = thin_film_velocities_lagrangian(
        r, z, phi, p
    )

    kinetic = 0.5 * (vr * vr + vz * vz)

    if p.use_vdw:
        V_vdw = vdw_energy_term(z, p)
    else:
        V_vdw = np.zeros_like(z)

    V_opt, D_t, I_local = optical_energy_terms(t, r, z, phi, D, p)

    r_t = vr
    z_t = vz

    phi_t = kinetic + V_vdw + V_opt - p.phi_damping * phi

    if p.use_delay:
        return pack_state(r_t, z_t, phi_t, D_t)

    return pack_state(r_t, z_t, phi_t)


def rk4_step(f, t: float, y: Array, dt: float, p: ThinFilmParams) -> Array:
    k1 = f(t, y, p)
    k2 = f(t + 0.5 * dt, y + 0.5 * dt * k1, p)
    k3 = f(t + 0.5 * dt, y + 0.5 * dt * k2, p)
    k4 = f(t + dt, y + dt * k3, p)

    return y + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)


def integrate_rk4(
    y0: Array,
    p: ThinFilmParams,
    t0: float,
    t1: float,
    dt: float,
    store_every: int = 1,
):
    steps = int(np.ceil((t1 - t0) / dt))

    ys = []
    ts = []

    t = t0
    y = y0.copy()

    for step in range(steps + 1):
        if step % store_every == 0:
            ys.append(y.copy())
            ts.append(t)

        if step == steps:
            break

        actual_dt = min(dt, t1 - t)
        y = rk4_step(rhs_thin_film_lagrangian, t, y, actual_dt, p)
        t += actual_dt

    return np.array(ts), np.array(ys)

In [43]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import j0, jn_zeros

N = 256
L = 1e-3 # m
h0 = 15e-9 # m

## adimensionalization length scale: R
h0 = h0 / L
R = 1.0


dr = R / N
r0 = (np.arange(N) + 0.5) * dr
z0 = np.zeros_like(r0)

mode = 0
beta = jn_zeros(0, mode + 1)[mode]

# Small initial potential perturbation.
phi0 = 0.0* 1e-4 * j0(beta * r0)

# Delayed intensity state, same size as r.
D0 = np.zeros_like(r0)

p = ThinFilmParams(
    R=R,
    h0=h0,
    side_bc="dirichlet",
    phi_wall=0.0,

    use_vdw=True,
    vdw_coeff=1.0,

    use_delay=True,
    tau_delay=0.05,

    intensity0=10.0,
    intensity_amp=0.0,
    omega_drive=2.0 * np.pi * 1.0,
    phase_drive=0.0,

    profile_center=0.5,
    profile_sigma=0.05,

    instant_coeff=0.0,
    delayed_coeff=1e-4,

    phi_damping=0.0,
)

y0 = pack_state(r0, z0, phi0, D0)

ts, ys = integrate_rk4(
    y0,
    p,
    t0=0.0,
    t1=1.0,
    dt=1e-5,
    store_every=100,
)

# Unpack final state.
r, z, phi, D = unpack_state(ys[-1], N, p.use_delay)

plt.figure()
plt.plot(r, z * L)
plt.xlabel("r")
plt.ylabel("z (m)")
plt.title("Final surface displacement")
plt.show()

In [45]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider


def plot_thin_film_states_slider(ts, ys, p, *, show_velocities=True):
    """
    Interactive viewer for states produced by integrate_rk4.

    Assumes:
        ys[m] = [r, z, phi] or [r, z, phi, D]
    depending on p.use_delay.
    """

    if p.use_delay:
        N = ys.shape[1] // 4
    else:
        N = ys.shape[1] // 3

    nrows = 4 if show_velocities else 2
    if p.use_delay:
        nrows += 1

    fig, axes = plt.subplots(nrows, 1, figsize=(10, 2.6 * nrows), sharex=True)
    if nrows == 1:
        axes = [axes]

    plt.subplots_adjust(bottom=0.12)

    # Initial state
    idx0 = 0
    r, z, phi, D = unpack_state(ys[idx0], N, p.use_delay)

    lines = {}

    ax = axes[0]
    lines["z"], = ax.plot(r, z, label="z")
    ax.set_ylabel("z")
    ax.legend()
    ax.grid(True)

    ax = axes[1]
    lines["phi"], = ax.plot(r, phi, label="phi")
    ax.set_ylabel("phi")
    ax.legend()
    ax.grid(True)

    row = 2

    if p.use_delay:
        ax = axes[row]
        lines["D"], = ax.plot(r, D, label="D")
        ax.set_ylabel("D")
        ax.legend()
        ax.grid(True)
        row += 1

    if show_velocities:
        vr, vz, q, z_prime, div_flux, F = thin_film_velocities_lagrangian(r, z, phi, p)

        ax = axes[row]
        lines["vr"], = ax.plot(r, vr, label="vr")
        ax.set_ylabel("vr")
        ax.legend()
        ax.grid(True)
        row += 1

        ax = axes[row]
        lines["vz"], = ax.plot(r, vz, label="vz")
        lines["q"], = ax.plot(r, q, "--", label="q")
        ax.set_ylabel("vz / q")
        ax.set_xlabel("r")
        ax.legend()
        ax.grid(True)

    fig.suptitle(f"state {idx0}/{len(ts)-1}, t = {ts[idx0]:.6g}")

    # Slider axis
    slider_ax = fig.add_axes([0.15, 0.035, 0.7, 0.03])
    slider = Slider(
        ax=slider_ax,
        label="state",
        valmin=0,
        valmax=len(ts) - 1,
        valinit=idx0,
        valstep=1,
    )

    def autoscale_all():
        for ax in axes:
            ax.relim()
            ax.autoscale_view()

    def update(val):
        idx = int(slider.val)

        r, z, phi, D = unpack_state(ys[idx], N, p.use_delay)

        lines["z"].set_data(r, z * L)
        lines["phi"].set_data(r, phi)

        if p.use_delay:
            lines["D"].set_data(r, D)

        if show_velocities:
            vr, vz, q, z_prime, div_flux, F = thin_film_velocities_lagrangian(r, z, phi, p)

            lines["vr"].set_data(r, vr)
            lines["vz"].set_data(r, vz)
            lines["q"].set_data(r, q)

        fig.suptitle(f"state {idx}/{len(ts)-1}, t = {ts[idx]:.6g}")

        autoscale_all()
        fig.canvas.draw_idle()

    slider.on_changed(update)

    autoscale_all()
    plt.show()

    return fig, slider

In [46]:
fig, slider = plot_thin_film_states_slider(ts, ys, p, show_velocities=True)